## **Tensorflow Keras ka use karke Optimization**
HackerRank ke is scenario mein aap **Optimisation** ke baare mein seekhenge, aur alag-alag optimizers kaise use kiye jaate hain ye samjhenge.

Hum yahan teen scenarios dekhenge:
1. Alag-alag optimisation algorithms ko tune karna,
2. SGD optimizer ke learning rate aur momentum ko tune karna
3. Adam optimizer ke beta values ko tune karna

**Note** - Challenge complete karne ke baad aakhir mein sabhi cells ko restart karke run zaroor karein

### Zaroori packages import karein

In [ ]:
import numpy as np                                       # numpy - numerical operations (arrays, seed set karne) ke liye
from sklearn.model_selection import GridSearchCV          # GridSearchCV - hyperparameters ki alag-alag combinations test karne ke liye
from keras.models import Sequential                       # Sequential - layer-by-layer neural network model banane ke liye
from keras.layers import Dense                             # Dense - fully connected (har node se juda) layer banane ke liye
from keras.wrappers.scikit_learn import KerasClassifier    # KerasClassifier - Keras model ko sklearn ke saath compatible banata hai (GridSearchCV use karne ke liye)
from sklearn.datasets import load_iris                     # load_iris - built-in Iris flower dataset load karne ke liye
from sklearn.model_selection import train_test_split       # train_test_split - data ko train/test mein split karne ke liye (yahan directly use nahi ho raha)
from sklearn.preprocessing import LabelEncoder,StandardScaler  # LabelEncoder/StandardScaler - labels encode aur data scale karne ke liye
from sklearn.utils import shuffle                          # shuffle - data ki rows ko randomly mix karne ke liye
from keras.utils.np_utils import to_categorical            # to_categorical - class labels ko one-hot encoded format mein convert karta hai
from keras.optimizers import SGD,Adam                       # SGD aur Adam - do optimization algorithms jo model train karte waqt use honge
from matplotlib import pyplot                               # pyplot - graphs/plots banane ke liye
import seaborn as sns                                        # seaborn - matplotlib ke upar bana hua, sundar statistical plots ke liye
import pandas as pd                                           # pandas - tabular data (DataFrame) handle karne ke liye
from keras.models import model_from_json                      # model_from_json - JSON se saved model structure wapas load karne ke liye


### Dataset Load Karein

- `load_iris()` function use karke iris dataset load karein.
- Iris dataset ka data variable **X** mein store karein.
- Iris dataset ka target variable **y** mein store karein.
- **y** ko `to_categorical` function se categorical variable mein convert karke wapas **y** mein save karein.
- **seed** variable mein seed value 7 set karein aur numpy ke `random.seed` function se seed value set karein.
- Ab **X** aur **y** data ko `shuffle` function se shuffle karke **X**, **Y** variables mein save karein.

In [ ]:
# fix random seed for reproducibility (baar-baar same random result aane ke liye seed fix karte hain)

iris = load_iris()                          # load_iris() - sklearn ka built-in Iris flower dataset load karta hai
X = iris.data                               # X mein dataset ke saare input features (4 columns: sepal/petal length-width) store honge
#Y = to_categorical(iris.target,3)
y = iris.target                             # y mein dataset ke original class labels (0,1,2) store honge
y = to_categorical(y, 3)                    # to_categorical - labels ko one-hot encoding mein convert karta hai (3 classes ke liye)
seed = 7                                    # seed variable mein fix value 7 set kar rahe hain reproducibility ke liye
np.random.seed(seed)                        # numpy ka random seed fix kar rahe hain taaki result har baar same aaye

X, Y = shuffle(X, y)                        # shuffle - X aur y data ki rows ko ek saath randomly mix karta hai


---------------------------------------------------------------------------
## **1. Alag-Alag Optimisation Algorithms Tune Karna**
-------------------------------------------------------------------

**optimizer** variable mein neeche diye gaye optimizers ko list ke roop mein pass karein -
- SGD, RMSprop, Adam, Nadam

**param_grid** mein `optimizer` parameter ko dict ke through pass karein

In [ ]:
optimizer = ['SGD', 'RMSprop', 'Adam', 'Nadam']    # optimizer - test karne wale 4 optimization algorithms ki list

param_grid = dict(optimizer=optimizer)              # param_grid - GridSearchCV ko batata hai konsa parameter (optimizer) tune karna hai


### Model Banayein

`create_model` function mein Dense class use karke ek fully-connected network structure banayein
- Ek Sequential model banayein
- Model 4 variables wali rows expect karta hai (input_dim=4 argument)
- Pehli hidden layer mein 64 nodes hain aur relu activation function use hota hai.
- Doosri hidden layer mein 32 nodes hain aur relu activation function use hota hai.
- Teesri hidden layer mein 16 nodes hain aur relu activation function use hota hai.
- Output layer mein 3 nodes hain aur softmax activation function use hota hai.
- Model compile karte waqt neeche diye parameters pass karein -

           - optimizer as optimizer
           - loss as categorical cross entropy
           - metrics as accuracy.
 - Compiled model return karein

In [ ]:
def create_model(optimizer='adam'):
    model = Sequential()                                    # Sequential() - ek khaali, layer-by-layer model create karta hai
    model.add(Dense(64, input_dim=4, activation='relu'))    # Pehli hidden layer - 64 nodes, input 4 features leta hai, relu activation
    model.add(Dense(32, activation='relu'))                  # Doosri hidden layer - 32 nodes, relu activation
    model.add(Dense(16, activation='relu'))                  # Teesri hidden layer - 16 nodes, relu activation
    model.add(Dense(3, activation='softmax'))                # Output layer - 3 nodes (3 classes), softmax probability deta hai
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])  # Model ko diye optimizer, loss aur accuracy metric ke saath compile karna
    return model                                              # Bana hua compiled model wapas bhejna


`KerasClassifier` function use karke neeche diye parameters ke saath model function ko call karein -
- build_fn as create_model
- batch_size as 10
- verbose as 0
- epochs as 10

Upar wale ko **model** variable mein save karein

In [ ]:
model = KerasClassifier(build_fn=create_model, batch_size=10, verbose=0, epochs=10)
# KerasClassifier - hamare create_model function ko sklearn-compatible classifier mein wrap karta hai
# build_fn -> kaunsa model function call hoga, batch_size -> ek baar mein kitni samples se seekhega,
# verbose=0 -> training ka log print nahi hoga, epochs -> poore dataset par kitni baar training hogi


**grid** mein `GridSearchCV` function use karein aur neeche diye parameters pass karein -
- estimator as model
- param_grid as param_grid
- n_jobs as 1

Ab **grid** use karke model ko X aur Y ke saath fit karein aur **grid_result** mein save karein

**Note** - fit model run karte waqt is cell ko chalne mein 2-5 minute lag sakte hain

In [ ]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1)
# GridSearchCV - diye gaye param_grid ke sabhi combinations try karke best optimizer dhundhta hai
grid_result = grid.fit(X, Y)
# grid.fit(X, Y) - actual mein training/searching chalu karta hai aur result grid_result mein store hota hai


### Results ko summarize karne ke liye neeche wali cell run karein

In [ ]:
# summarize results (results ka summary print karna)
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
# grid_result.best_score_ -> sabse achha (highest) score, best_params_ -> wo optimizer jisse best score mila
means = grid_result.cv_results_['mean_test_score']    # means -> har optimizer ke liye average test score
stds = grid_result.cv_results_['std_test_score']      # stds -> har optimizer ke score ka standard deviation (variation)
params = grid_result.cv_results_['params']            # params -> har combination mein use hue parameters ki list
for mean, stdev, param in zip(means, stds, params):   # teeno list ko ek saath loop karke print karna
    print("%f (%f) with: %r" % (mean, stdev, param))


### Results plot karne ke liye neeche wali cell run karein

In [ ]:
pyplot.figure(figsize=(10,8))                          # figure(figsize) - plot ka size (width, height) set karna
#pyplot.xticks(grid_result1.cv_results_['mean_test_score'])
pyplot.title("Performance metrics of each optimiser")  # graph ka title set karna
plot=sns.barplot(grid_result.cv_results_['mean_test_score'],optimizer)
# sns.barplot - har optimizer (y-axis) ke against uska mean_test_score (x-axis) bar chart mein dikhata hai
pyplot.show()                                            # tayar plot screen par dikhana


---------------------------------------------------------------------------
## **2. SGD Optimizer ka Learning Rate aur Momentum Tune Karna**
-------------------------------------------------------------------
**learn_rate** variable mein neeche diye learn rates ko list ke roop mein pass karein -
- 0.001, 0.01, 0.3

**momentum** variable mein neeche diye momentums ko list ke roop mein pass karein -
- 0.0, 0.4, 0.9

In [ ]:
learn_rate = [0.001, 0.01, 0.3]    # learn_rate - test karne wali alag-alag learning rate values ki list
momentum = [0.0, 0.4, 0.9]         # momentum - test karne wali alag-alag momentum values ki list


### Model Banayein

`create_model1` function mein upar jaise hi same model parameters use karke model banayein

**optimizer** variable mein SGD optimizer ka use karke neeche diye parameters pass karein

     - lr as learn_rate
     - momentum as momentum

Model compile karte waqt neeche diye parameters pass karein -

     - optimizer as optimizer
     - loss as categorical cross entropy
     - metrics as accuracy.

Compiled model return karein

In [ ]:
def create_model1(learn_rate=0.01, momentum=0):
    model = Sequential()                                    # Sequential() - naya empty model create karna
    model.add(Dense(64, input_dim=4, activation='relu'))    # Pehli hidden layer - 64 nodes, 4 input features, relu
    model.add(Dense(32, activation='relu'))                  # Doosri hidden layer - 32 nodes, relu
    model.add(Dense(16, activation='relu'))                  # Teesri hidden layer - 16 nodes, relu
    model.add(Dense(3, activation='softmax'))                # Output layer - 3 nodes, softmax
    optimizer = SGD(lr=learn_rate, momentum=momentum)
    # SGD optimizer banaya jisme di gayi learn_rate aur momentum values use hongi
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    # Model ko upar bane SGD optimizer, loss aur accuracy metric ke saath compile karna
    return model                                              # Compiled model wapas bhejna


`KerasClassifier` function use karke neeche diye parameters ke saath model function ko call karein -

- build_fn as create_model1
- batch_size as 10
- verbose as 0
- epochs as 10

Upar wale ko **model1** variable mein save karein

In [ ]:
model1 = KerasClassifier(build_fn=create_model1, batch_size=10, verbose=0, epochs=10)
# create_model1 function ko KerasClassifier ke through sklearn-compatible model1 banaya


**param_grid1** mein `learn_rate` parameter ko learn_rate aur `momentum` parameter ko momentum ke saath dict ke through pass karein

**grid1** mein `GridSearchCV` function use karein aur neeche diye parameters pass karein -
- estimator as model1
- param_grid as param_grid1
- n_jobs as 1

In [ ]:
param_grid1 = dict(learn_rate=learn_rate, momentum=momentum)
# param_grid1 - GridSearchCV ko batata hai ki learn_rate aur momentum dono tune karne hain
grid1 = GridSearchCV(estimator=model1, param_grid=param_grid1, n_jobs=1)
# grid1 - model1 par param_grid1 ki sabhi combinations test karega


numpy ke `random.seed` function se seed value set karein.

Ab **grid1** use karke model ko X aur Y ke saath fit karein aur **grid_result1** mein save karein

**Note** - fit model run karte waqt is cell ko chalne mein 2-5 minute lag sakte hain

In [ ]:
np.random.seed(seed)                 # numpy random seed dobara fix kar rahe hain reproducibility ke liye

grid_result1 = grid1.fit(X, Y)       # grid1.fit(X, Y) - learn_rate/momentum combinations par training/search chalu karna


### Results ko summarize karne ke liye neeche wali cell run karein

In [ ]:
# summarize results (results ka summary print karna)
print("Best: %f using %s" % (grid_result1.best_score_, grid_result1.best_params_))
# best_score_ aur best_params_ - sabse achha score aur uska learn_rate/momentum combination
means1 = grid_result1.cv_results_['mean_test_score']    # har combination ka average test score
stds1 = grid_result1.cv_results_['std_test_score']      # har combination ke score ka standard deviation
params1 = grid_result1.cv_results_['params']            # har combination ke actual parameters
for mean, stdev, param in zip(means1, stds1, params1):  # teeno ko saath loop karke print karna
    print("%f (%f) with: %r" % (mean, stdev, param))


### Results plot karne ke liye neeche wali cell run karein

In [ ]:
params1=pd.DataFrame(params1)                    # params ki list ko DataFrame (table) mein convert kar rahe hain
pyplot.figure(figsize=(10,8))                     # plot ka size set karna
#pyplot.xticks(grid_result1.cv_results_['mean_test_score'])
pyplot.title("Performance metrics of SGD optimiser with different learning rates and momentum")
plot1=sns.barplot(params1["learn_rate"],grid_result1.cv_results_['mean_test_score'],hue=params1["momentum"])
# x-axis par learn_rate, y-axis par mean_test_score, aur momentum ke hisaab se alag color (hue) dikhana
plot1.set(ylabel='Score')                          # y-axis ka label 'Score' set karna
pyplot.show()                                       # tayar plot dikhana


---------------------------------------------------------------------------
## **3. Adam Optimizer ke Beta Values Tune Karna**
-------------------------------------------------------------------

**beta_1** variable mein neeche diye values ko list ke roop mein pass karein -
- 0.001, 0.01, 0.3

**beta_2** variable mein neeche diye values ko list ke roop mein pass karein -
- 0.0, 0.4, 0.9

In [ ]:
beta_1 = [0.001, 0.01, 0.3]    # beta_1 - test karne wali alag-alag beta_1 values ki list
beta_2 = [0.0, 0.4, 0.9]       # beta_2 - test karne wali alag-alag beta_2 values ki list


### Model Banayein

`create_model2` function mein upar jaise hi same model parameters use karke model banayein

**optimizer** variable mein Adam optimizer ka use karke neeche diye parameters pass karein

     - beta_1 as beta_1
     - beta_2 as beta_2

Model compile karte waqt neeche diye parameters pass karein -

     - optimizer as optimizer
     - loss as categorical cross entropy
     - metrics as accuracy.

Compiled model return karein

In [ ]:
def create_model2(beta_1=0.01, beta_2=0):
    model = Sequential()                                    # naya empty Sequential model banana
    model.add(Dense(64, input_dim=4, activation='relu'))    # Pehli hidden layer - 64 nodes, 4 input features, relu
    model.add(Dense(32, activation='relu'))                  # Doosri hidden layer - 32 nodes, relu
    model.add(Dense(16, activation='relu'))                  # Teesri hidden layer - 16 nodes, relu
    model.add(Dense(3, activation='softmax'))                # Output layer - 3 nodes, softmax
    optimizer = Adam(beta_1=beta_1, beta_2=beta_2)
    # Adam optimizer banaya jisme di gayi beta_1 aur beta_2 values use hongi
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    # Model ko upar bane Adam optimizer, loss aur accuracy metric ke saath compile karna
    return model                                              # Compiled model wapas bhejna


`KerasClassifier` function use karke neeche diye parameters ke saath model function ko call karein -

- build_fn as create_model2
- batch_size as 10
- verbose as 0
- epochs as 10

Upar wale ko **model2** variable mein save karein

In [ ]:
model2 = KerasClassifier(build_fn=create_model2, batch_size=10, verbose=0, epochs=10)
# create_model2 function ko KerasClassifier ke through sklearn-compatible model2 banaya


**param_grid2** mein `beta_1` parameter ko beta_1 aur `beta_2` parameter ko beta_2 ke saath dict ke through pass karein

**grid2** mein `GridSearchCV` function use karein aur neeche diye parameters pass karein -
- estimator as model2
- param_grid as param_grid2
- n_jobs as 1

In [ ]:
param_grid2 = dict(beta_1=beta_1, beta_2=beta_2)
# param_grid2 - GridSearchCV ko batata hai ki beta_1 aur beta_2 dono tune karne hain
grid2 = GridSearchCV(estimator=model2, param_grid=param_grid2, n_jobs=1)
# grid2 - model2 par param_grid2 ki sabhi combinations test karega


numpy ke `random.seed` function se seed value set karein.

Ab **grid2** use karke model ko X aur Y ke saath fit karein aur **grid_result2** mein save karein

**Note** - fit model run karte waqt is cell ko chalne mein 2-5 minute lag sakte hain

In [ ]:
np.random.seed(seed)                 # numpy random seed dobara fix kar rahe hain reproducibility ke liye

grid_result2 = grid2.fit(X, Y)       # grid2.fit(X, Y) - beta_1/beta_2 combinations par training/search chalu karna


### Results ko summarize karne ke liye neeche wali cell run karein

In [ ]:
# summarize results (results ka summary print karna)
print("Best: %f using %s" % (grid_result2.best_score_, grid_result2.best_params_))
# best_score_ aur best_params_ - sabse achha score aur uska beta_1/beta_2 combination
means2 = grid_result2.cv_results_['mean_test_score']    # har combination ka average test score
stds2 = grid_result2.cv_results_['std_test_score']      # har combination ke score ka standard deviation
params2 = grid_result2.cv_results_['params']            # har combination ke actual parameters
for mean, stdev, param in zip(means2, stds2, params2):  # teeno ko saath loop karke print karna
    print("%f (%f) with: %r" % (mean, stdev, param))


### Results plot karne ke liye neeche wali cell run karein

In [ ]:
params2=pd.DataFrame(params2)                    # params ki list ko DataFrame (table) mein convert kar rahe hain
pyplot.figure(figsize=(10,8))                     # plot ka size set karna
#pyplot.xticks(grid_result1.cv_results_['mean_test_score'])
pyplot.title("Performance metrics of SGD optimiser with different beta values")
plot2=sns.barplot(params2["beta_1"],grid_result2.cv_results_['mean_test_score'],hue=params2["beta_2"])
# x-axis par beta_1, y-axis par mean_test_score, aur beta_2 ke hisaab se alag color (hue) dikhana
plot2.set(ylabel='Score')                          # y-axis ka label 'Score' set karna
pyplot.show()                                       # tayar plot dikhana


**Note**

Isi tarah aap doosre optimization algorithms bhi use karke operations perform kar sakte hain.

Jaise humne is exercise mein kiya, waise hi aap doosre optimizers ke parameters bhi tune kar sakte hain.

### Apne scores aur model ko testing ke liye save karne ke liye neeche wali cells run karein

In [ ]:
with open("score.txt","w") as f:                       # score.txt file ko write mode mein khol rahe hain
    f.write(str(round(grid_result.best_score_,2)))     # pehle scenario ka best score (2 decimal tak round) file mein likhna
with open("params.txt","w") as f:                      # params.txt file ko write mode mein khol rahe hain
    f.write(str(grid_result.best_params_))              # pehle scenario ke best parameters file mein likhna

with open("score1.txt","w") as f:                      # score1.txt file khol rahe hain
    f.write(str(round(grid_result1.best_score_,2)))    # doosre scenario ka best score likhna
with open("params1.txt","w") as f:                     # params1.txt file khol rahe hain
    f.write(str(grid_result1.best_params_))             # doosre scenario ke best parameters likhna
    
with open("score2.txt","w") as f:                      # score2.txt file khol rahe hain
    f.write(str(round(grid_result2.best_score_,2)))    # teesre scenario ka best score likhna
with open("params2.txt","w") as f:                     # params2.txt file khol rahe hain
    f.write(str(grid_result2.best_params_))             # teesre scenario ke best parameters likhna


In [ ]:
def save_model(model):
    # saving model (model ka structure save karna)
    json_model = model.to_json()                 # model.to_json() - model ke structure (layers, config) ko JSON string mein convert karna
    open('model.json', 'w').write(json_model)    # is JSON string ko model.json file mein likh dena
    # saving weights (model ke seekhe hue weights save karna)
    model.save_weights('model.h5', overwrite=True)   # model ke trained weights ko model.h5 file mein save karna
classifier=create_model()             # ek naya model instance create_model() function se banana
save_model(classifier)                # upar banaye function se model save karna
